# KneeWise — Exploratory Data Analysis (EDA)
### AI-Powered Knee Osteoarthritis Screening & Severity Grading Platform

This notebook performs exploratory analysis of the **Mendeley Knee Osteoarthritis Severity Grading Dataset** (Chen et al., OAI cohort).

**Objectives:**
1. Verify hardware acceleration (NVIDIA RTX 3050 GPU, 4 GB VRAM).
2. Inspect dataset composition, partition sizes, and class balance across Kellgren–Lawrence (KL) grades 0–4.
3. Audit patient identifiers to safeguard against cross-split data leakage.
4. Visualize representative radiographs and inspect pixel intensity distributions.

In [ ]:
import os
import sys
from pathlib import Path

# Ensure matplotlib temporary cache works in restricted environments
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

# Add src directory to system path
PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from config import (
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    EDA_OUTPUT_DIR,
    FULL_MANIFEST_CSV,
    TRAIN_MANIFEST_CSV,
    VAL_MANIFEST_CSV,
    TEST_MANIFEST_CSV,
    KL_CLASSES,
    KL_GRADE_DESCRIPTIONS,
    KL_GRADE_SHORT_LABELS,
    get_device,
)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Device: {get_device()}")

## 1. Hardware & CUDA Environment Inspection

In [ ]:
from verify_gpu import verify_gpu
verify_gpu()

## 2. Load Processed Manifests
Manifests are pre-built by `src/preprocess.py` without duplicating raw image files.

In [ ]:
if FULL_MANIFEST_CSV.exists():
    df = pd.read_csv(FULL_MANIFEST_CSV)
    print(f"Loaded full dataset manifest: {len(df):,} total images.")
    print(df.head())
else:
    print(f"Manifest not yet generated at {FULL_MANIFEST_CSV}.")
    print("Run: python src/preprocess.py after extracting dataset into data/raw/")

## 3. Kellgren–Lawrence (KL) Grade Class Distribution
Osteoarthritis severity is graded clinically from Grade 0 (No OA) to Grade 4 (Severe OA).

In [ ]:
if 'df' in locals() and not df.empty:
    class_counts = df['kl_grade'].value_counts().sort_index()
    total = len(df)
    
    summary_data = []
    for grade in KL_CLASSES:
        count = class_counts.get(grade, 0)
        pct = (count / total * 100) if total > 0 else 0
        summary_data.append({
            "KL Grade": grade,
            "Description": KL_GRADE_DESCRIPTIONS[grade],
            "Count": count,
            "Percentage": f"{pct:.2f}%",
        })
    summary_df = pd.DataFrame(summary_data)
    display(summary_df)
    
    # Plot
    plt.figure(figsize=(9, 5))
    colors = ["#2b5c8f", "#4682b4", "#5c9ebb", "#e07a5f", "#c0392b"]
    bars = plt.bar([f"Grade {g}" for g in KL_CLASSES], [class_counts.get(g, 0) for g in KL_CLASSES], color=colors, edgecolor="black")
    plt.title("Kellgren-Lawrence Grade Distribution in Dataset", fontsize=13, fontweight="bold")
    plt.ylabel("Image Count")
    plt.grid(axis="y", alpha=0.3)
    plt.show()

## 4. Visualizing Representative Radiographs Across KL Grades

In [ ]:
if 'df' in locals() and not df.empty:
    fig, axes = plt.subplots(1, 5, figsize=(18, 4.5))
    for i, grade in enumerate(KL_CLASSES):
        subset = df[df["kl_grade"] == grade]
        if not subset.empty:
            img_path = Path(subset.iloc[0]["image_path"])
            with Image.open(img_path) as img:
                axes[i].imshow(img.convert("L"), cmap="bone")
                axes[i].set_title(KL_GRADE_SHORT_LABELS[grade], fontweight="bold")
        axes[i].axis("off")
    plt.suptitle("Representative Knee Radiographs Across KL Severity Grades", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 5. Patient Data Leakage Audit
Verifies that no patient identifier is shared across Train, Validation, and Test splits.

In [ ]:
if TRAIN_MANIFEST_CSV.exists() and VAL_MANIFEST_CSV.exists() and TEST_MANIFEST_CSV.exists():
    train_df = pd.read_csv(TRAIN_MANIFEST_CSV)
    val_df = pd.read_csv(VAL_MANIFEST_CSV)
    test_df = pd.read_csv(TEST_MANIFEST_CSV)
    
    train_pts = set(train_df["patient_id"])
    val_pts = set(val_df["patient_id"])
    test_pts = set(test_df["patient_id"])
    
    leakage_tv = train_pts.intersection(val_pts)
    leakage_tt = train_pts.intersection(test_pts)
    leakage_vt = val_pts.intersection(test_pts)
    
    print(f"Unique Train Patients:      {len(train_pts):,}")
    print(f"Unique Validation Patients: {len(val_pts):,}")
    print(f"Unique Test Patients:       {len(test_pts):,}")
    print(f"Train-Val Leakage:          {len(leakage_tv)} patients")
    print(f"Train-Test Leakage:         {len(leakage_tt)} patients")
    print(f"Val-Test Leakage:           {len(leakage_vt)} patients")
    
    if not (leakage_tv or leakage_tt or leakage_vt):
        print("\n[PASSED] Strict patient-level isolation confirmed. Zero cross-split leakage.")
    else:
        print("\n[WARNING] Patient overlap detected! Re-run preprocess.py with --force_patient_split.")